# recurrentlens 02 — Explore an SAE

Inspect features of a sparse autoencoder over Mamba-130M activations.

> **v0.1.0.post1 note:** Pretrained Mamba-130M SAEs ship in **v0.1.1** (trained via [`notebooks/03_train_mamba130m_sae.ipynb`](./03_train_mamba130m_sae.ipynb) on Colab T4). Until then, this notebook falls back to an *untrained* TopK SAE to demonstrate the public API end-to-end. Replace `REPO_ID` below once v0.1.1 artifacts land.

In [ ]:
%pip install -q recurrentlens[mamba] || %pip install -q git+https://github.com/hinanohart/recurrentlens.git

In [ ]:
import torch
import recurrentlens as rl

# Until v0.1.1 ships the pretrained artifacts, leave REPO_ID = None and the
# notebook will use a fresh (untrained) TopK SAE for the API demo.
REPO_ID = None  # e.g. 'hinanohart/recurrentlens-mamba130m-L6-sae' (planned v0.1.1)
LAYER = 6
HOOK_SITE = 'out_proj_out'

In [ ]:
model = rl.load_model('state-spaces/mamba-130m-hf')

if REPO_ID is not None:
    sae = rl.hub.load_sae(REPO_ID)
    print('Loaded pretrained SAE:', sae)
else:
    from recurrentlens.sae.variants import TopKSAE
    sae = TopKSAE(
        d_in=model.d_model, d_sae=2048, k=16,
        hook_site=HOOK_SITE, layer=LAYER,
        model_id=model.model_id,
    )
    print('Using an UNTRAINED TopK SAE for API demo. Set REPO_ID after v0.1.1 lands.')
    print(sae)

In [ ]:
# Collect activations from a small corpus.
from recurrentlens.sae.cache import ActivationCache
from recurrentlens.hooks.registry import HookManager

texts = [
    'The capital of France is Paris.',
    'Quicksort partitions an array around a pivot.',
    'Mamba uses selective state-space mixing.',
    'In thermodynamics, entropy never decreases.',
]
cache = ActivationCache(capacity=4096, d_in=model.d_model, backend='ram')
ctx_per_token = []
with HookManager(model) as mgr:
    h = mgr.add(site=sae.hook_site, layer=sae.layer, transform=lambda t: t.detach().cpu())
    for t in texts:
        ids = model.tokenizer(t, return_tensors='pt').input_ids.to(model.device)
        with torch.no_grad():
            _ = model.forward(ids)
        for tok_id in ids[0].tolist():
            ctx_per_token.append(model.tokenizer.decode([tok_id]))
        cache.append(h.activations[-1].reshape(-1, model.d_model).numpy())
        h.clear()

In [ ]:
# Render top-k activating tokens for a feature.
import torch
acts = torch.as_tensor(cache._store[: len(cache)])
out = rl.viz.feature_explorer(sae, feature_id=42, activations=acts, contexts=ctx_per_token, top_k=10)
out  # renders inline in Jupyter

In [ ]:
# Evaluate the SAE on these activations (recon_mse / l0 / fvu).
report = rl.bench.evaluate(sae, activations=acts)
print(report)

In [ ]:
# Steer a generation by amplifying a feature.
# With an untrained SAE this is a sanity check; with v0.1.1 pretrained
# weights it produces semantically meaningful behavior changes.
ids = model.tokenizer('In the year 2030,', return_tensors='pt').input_ids.to(model.device)
with rl.steer(model, sae, feature_id=42, vector_scale=3.0):
    out = model.hf_model.generate(ids, max_new_tokens=20)
print(model.tokenizer.decode(out[0]))